# Forecast To Inventory Policy

Forecasts are converted to safety stock, reorder point, min/max, order quantity, and pallet-position impact.

In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import Image, Markdown, display
except Exception:
    def display(value):
        print(value)
    def Markdown(text):
        return text
    class Image:
        def __init__(self, filename=None, **kwargs):
            self.filename = filename
        def __repr__(self):
            return f"Image(filename={self.filename!r})"

ROOT = Path.cwd()
if ROOT.name != "v7_rm_pm_forecast_planning":
    ROOT = Path("Ai miroservices/modeling/v7_rm_pm_forecast_planning").resolve()
OUT = ROOT / "outputs"
PLOTS = OUT / "plots"
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)

def load_csv(name, **kwargs):
    path = OUT / name
    if not path.exists():
        raise FileNotFoundError(f"Missing artifact: {path}. Run PYTHONPATH=. python3 -m pipeline.run_all first.")
    return pd.read_csv(path, **kwargs)

def load_json(name):
    path = OUT / name
    if not path.exists():
        raise FileNotFoundError(f"Missing artifact: {path}. Run PYTHONPATH=. python3 -m pipeline.run_all first.")
    return json.loads(path.read_text())

def show_plot(name):
    path = PLOTS / name
    if path.exists():
        display(Image(filename=str(path)))
    else:
        display(Markdown(f"Plot not generated: `{path}`"))

In [2]:
forecasts = load_csv('forecast_results_v7.csv', parse_dates=['forecast_period'])
forecasts.head(30)

,material_id,material_code,description,material_type,warehouse_id,warehouse_code,forecast_period,horizon,model_name,forecast_p10,forecast_p50,forecast_p90,method
0,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,7262019d-9bf4-4824-997c-d7b5c9158ef3,WH-001,2026-02-01,1,V7_RM_PM_DIRECT,37.91,77.25,180.96,lightgbm_global_rm_pm
1,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,7262019d-9bf4-4824-997c-d7b5c9158ef3,WH-001,2026-03-01,2,V7_RM_PM_DIRECT,41.51,84.58,198.15,lightgbm_global_rm_pm
2,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,7262019d-9bf4-4824-997c-d7b5c9158ef3,WH-001,2026-04-01,3,V7_RM_PM_DIRECT,46.93,95.63,224.04,lightgbm_global_rm_pm
3,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,7262019d-9bf4-4824-997c-d7b5c9158ef3,WH-001,2026-05-01,4,V7_RM_PM_DIRECT,43.61,88.87,208.20,lightgbm_global_rm_pm
4,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,7262019d-9bf4-4824-997c-d7b5c9158ef3,WH-001,2026-06-01,5,V7_RM_PM_DIRECT,50.14,102.18,239.37,lightgbm_global_rm_pm
5,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,7262019d-9bf4-4824-997c-d7b5c9158ef3,WH-001,2026-07-01,6,V7_RM_PM_DIRECT,39.03,79.53,186.32,lightgbm_global_rm_pm
6,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,7262019d-9bf4-4824-997c-d7b5c9158ef3,WH-001,2026-08-01,7,V7_RM_PM_DIRECT,29.82,60.78,142.38,lightgbm_global_rm_pm
7,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,7262019d-9bf4-4824-997c-d7b5c9158ef3,WH-001,2026-09-01,8,V7_RM_PM_DIRECT,29.93,60.98,142.86,lightgbm_global_rm_pm
8,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,7262019d-9bf4-4824-997c-d7b5c9158ef3,WH-001,2026-10-01,9,V7_RM_PM_DIRECT,24.78,50.49,118.29,lightgbm_global_rm_pm
9,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,7262019d-9bf4-4824-997c-d7b5c9158ef3,WH-001,2026-11-01,10,V7_RM_PM_DIRECT,20.80,42.39,99.31,lightgbm_global_rm_pm


In [3]:
policy = load_csv('inventory_policy_recommendations_v7.csv')
policy.head(30)

,material_id,material_code,description,material_type,horizon_months,forecast_p10_sum,forecast_p50_sum,forecast_p90_sum,avg_monthly_p50,on_hand_qty,available_qty,current_min_stock,current_max_stock,current_reorder_point,inventory_lead_time_days,units_per_pallet,weight_kg,volume_cm3,storage_type,abc_class,fms_class,intermittency_flag,planning_priority,nonzero_rate,cv,lead_time_days,daily_p50,daily_sigma,lead_demand,safety_stock,proposed_min_stock,proposed_reorder_point,proposed_target_stock,proposed_max_stock,suggested_order_qty,current_pallet_positions,target_pallet_positions,pallet_positions_delta,recommendation_status,rationale
0,5571d3cd-666f-4673-a41e-efeb313da005,100036,CAUSTIC SODA,raw_material,12,423293.33,862563.22,2020707.99,71880.268333,1000.0,1000.0,1347.38,77592.04,21037.24,30.0,1.0,12.0,1000.0,pallet,A,F,False,high_access_candidate,1.0,0.292674,30.0,2396.008944,1731.141910,71880.268333,15645.060329,15645.06,87525.33,878208.28,2036353.05,877208.28,1000.00,2036353.05,2035353.05,HIGH_RISK_REVIEW,"v7 direct RM/PM forecast, ABC=A, FMS=F, horizo..."
1,ed5d69d5-7370-4a93-b888-2aa711897187,101054,CALCIUM CARBONATE ( GROUND ),raw_material,12,283574.63,577852.36,1353722.06,48154.363333,1000.0,1000.0,40662.75,254847.39,142635.06,80.0,14.0,12.0,1000.0,pallet,A,F,False,high_access_candidate,1.0,0.326836,80.0,1605.145444,1159.734609,128411.635556,17115.419784,17115.42,145527.06,594967.78,1370837.48,593967.78,71.43,97916.96,97845.53,HIGH_RISK_REVIEW,"v7 direct RM/PM forecast, ABC=A, FMS=F, horizo..."
2,861bf563-9747-4508-96c0-5a0c976acbcd,101293,FLUFF UNTREATED - GOLDEN ISLES G4881,raw_material,12,261355.10,532574.66,1247651.05,44381.221667,1314.0,1314.0,4013.32,152770.98,101015.49,75.0,20.0,12.0,1000.0,pallet,A,F,False,high_access_candidate,1.0,0.335715,75.0,1479.374056,1068.863519,110953.054167,15273.438852,15273.44,126226.49,547848.10,1262924.49,546534.10,65.70,63146.22,63080.52,HIGH_RISK_REVIEW,"v7 direct RM/PM forecast, ABC=A, FMS=F, horizo..."
3,44dd0de1-d770-4f74-bccb-e6fc16f574ca,100098,SORBITOL,raw_material,12,256027.19,521717.77,1222216.78,43476.480833,1080.0,1080.0,36986.31,273141.09,205744.40,90.0,17.0,12.0,1000.0,pallet,A,F,False,high_access_candidate,1.0,0.296887,90.0,1449.216028,1047.073959,130429.442500,16390.136019,16390.14,146819.58,538107.91,1238606.92,537027.91,63.53,72859.23,72795.70,HIGH_RISK_REVIEW,"v7 direct RM/PM forecast, ABC=A, FMS=F, horizo..."
4,3670333e-585b-4f10-9d15-912dcb65d820,100108,TALCUM POWDER,raw_material,12,121867.62,248334.96,581768.89,20694.580000,1000.0,1000.0,30613.09,133723.39,36779.07,45.0,20.0,12.0,1000.0,pallet,A,F,False,high_access_candidate,1.0,0.476880,45.0,689.819333,498.401813,31041.870000,5516.578656,5516.58,36558.45,253851.54,587285.47,252851.54,50.00,29364.27,29314.27,HIGH_RISK_REVIEW,"v7 direct RM/PM forecast, ABC=A, FMS=F, horizo..."
5,584db23d-9317-457f-9e7d-d7592e043210,101580,SODIUM SILICATE,raw_material,12,104682.99,213317.08,499733.28,17776.423333,1200.0,1200.0,16226.41,86296.06,20468.81,30.0,20.0,12.0,1000.0,pallet,A,F,False,high_access_candidate,1.0,0.410001,30.0,592.547444,428.121846,17776.423333,3869.117878,3869.12,21645.54,217186.20,503602.40,215986.20,60.00,25180.12,25120.12,HIGH_RISK_REVIEW,"v7 direct RM/PM forecast, ABC=A, FMS=F, horizo..."
6,2b39aadc-6592-406c-914f-482f4cbb7ab5,100323,BC COLOGNE BULK - IMPORTED,raw_material,12,93161.99,189840.22,444734.55,15820.018333,1000.0,1000.0,4902.22,84031.63,54123.55,70.0,20.0,12.0,1000.0,pallet,A,F,False,high_access_candidate,1.0,0.229122,70.0,527.333944,381.004387,36913.376111,5259.723817,5259.72,42173.10,195099.94,449994.27,194099.94,50.00,22499.71,22449.71,HIGH_RISK_REVIEW,"v7 direct RM/PM forecast, ABC=A, FMS=F, horizo..."
7,9a11c556-2240-4f35-ae53-f0c8c8a88fb4,100460,GALAXY LES 70,raw_material,12,87291.02,177876.72,416707.93,14823.060000,1000.0,1000.0,182.02,5314.51,2509.90,60.0,20.0,12.0,1000.0,pallet,A,F,False,high_access_candidate,1.0,0.378986,60.0,494.102000,356.993981,29646.120000,4562.684753,4562.68,34208.8

In [4]:
policy.groupby(['recommendation_status', 'abc_class', 'fms_class']).size().reset_index(name='materials')

,recommendation_status,abc_class,fms_class,materials
0,APPLY_WITH_APPROVAL,B,F,40
1,APPLY_WITH_APPROVAL,C,F,178
2,APPLY_WITH_APPROVAL,C,M,11
3,APPLY_WITH_APPROVAL,C,S,29
4,HIGH_RISK_REVIEW,A,F,17
5,SAFE_TO_APPLY,C,M,4
6,SAFE_TO_APPLY,C,S,9


In [5]:
policy.sort_values("suggested_order_qty", ascending=False)[
    ["material_code", "description", "abc_class", "fms_class", "forecast_p50_sum", "available_qty", "proposed_reorder_point", "proposed_max_stock", "suggested_order_qty", "recommendation_status"]
].head(40)

,material_code,description,abc_class,fms_class,forecast_p50_sum,available_qty,proposed_reorder_point,proposed_max_stock,suggested_order_qty,recommendation_status
0,100036,CAUSTIC SODA,A,F,862563.22,1000.0,87525.33,2036353.05,877208.28,HIGH_RISK_REVIEW
1,101054,CALCIUM CARBONATE ( GROUND ),A,F,577852.36,1000.0,145527.06,1370837.48,593967.78,HIGH_RISK_REVIEW
2,101293,FLUFF UNTREATED - GOLDEN ISLES G4881,A,F,532574.66,1314.0,126226.49,1262924.49,546534.10,HIGH_RISK_REVIEW
3,100098,SORBITOL,A,F,521717.77,1080.0,146819.58,1238606.92,537027.91,HIGH_RISK_REVIEW
4,100108,TALCUM POWDER,A,F,248334.96,1000.0,36558.45,587285.47,252851.54,HIGH_RISK_REVIEW
5,101580,SODIUM SILICATE,A,F,213317.08,1200.0,21645.54,503602.40,215986.20,HIGH_RISK_REVIEW
6,100323,BC COLOGNE BULK - IMPORTED,A,F,189840.22,1000.0,42173.10,449994.27,194099.94,HIGH_RISK_REVIEW
7,100460,GALAXY LES 70,A,F,177876.72,1000.0,34208.80,421270.61,181439.40,HIGH_RISK_REVIEW
8,100050,GLYCERINE,A,F,111478.14,1120.0,11311.82,263179.35,112380.12,HIGH_RISK_REVIEW
9,100714,COCOMIDOPROPYL BETAINE,A,F,98509.75,1000.0,18945.15,233303.52,100036.61,HIGH_RISK_REVIEW
